# 02 — Strategy Construction & Weight Optimisation

**Objective:** Construct and compare portfolio weighting schemes for replicating CSI MARP 930929.

Strategies compared:
- **ERC_5:** Equal Risk Contribution, 5% annual vol target
- **RP_5:** Inverse-vol risk parity, 5% annual vol target
- **HRP_5:** Hierarchical Risk Parity (López de Prado 2016), 5% vol target
- **Equal:** 1/N equal weight
- **MARP_rep:** Ridge-regression replication of CSI MARP 930929

All optimisation is done on in-sample data only (2017–2021).

In [ ]:
%matplotlib inline
%config InlineBackend.figure_format = 'retina'
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

from pathlib import Path
import sys
sys.path.insert(0, str(Path.cwd().parent / 'src'))

from src.conventions import *
from src.optimizer import allocate_erc, allocate_vol_target, allocate_equal_weight, allocate_marp_replication

sns.set_style('whitegrid')
plt.rcParams.update({'figure.dpi': 120, 'font.size': 11})

In [ ]:
# Load data — in-sample period only
returns = pd.read_parquet(DATA_CLEAN / 'returns.parquet')
marp_full = pd.read_parquet(DATA_CLEAN / 'marp_official.parquet')

marp_date = pd.to_datetime(marp_full['日期'])
marp_s = pd.Series(marp_full['收盘'].values, index=marp_date).sort_index()
marp_ret = marp_s.pct_change().dropna()

assets = ['510300', '510500', '511010', '518880', '159980']
asset_labels = ['CSI 300', 'CSI 500', '5Y Treasury', 'Gold', 'Commodity']

# In-sample: 2017-2021
ret_is = returns.loc[:IS_END, assets].dropna()
marp_is = marp_ret.loc[:IS_END]
marp_is = marp_is.reindex(ret_is.index)

print(f'In-sample: {ret_is.index.min().date()} → {ret_is.index.max().date()}')
print(f'Trading days: {len(ret_is)}')
print(f'Assets: {ret_is.shape[1]}')

In [ ]:
# Compute all strategies on full in-sample period
print('Computing strategy weights...')

strategies = {}

# 1. ERC with vol target
strategies['ERC_5'] = allocate_erc(ret_is, vol_target=0.05)
print(f'  ERC_5    : {dict(zip(asset_labels, strategies["ERC_5"].round(4)))}')

# 2. Inverse vol risk parity
strategies['RP_5'] = allocate_vol_target(ret_is, vol_target=0.05)
print(f'  RP_5     : {dict(zip(asset_labels, strategies["RP_5"].round(4)))}')

# 3. Equal weight
strategies['Equal'] = allocate_equal_weight(ret_is)
print(f'  Equal     : {dict(zip(asset_labels, strategies["Equal"].round(4)))}')

# 4. MARP replication
strategies['MARP_rep'] = allocate_marp_replication(ret_is, marp_is, method='ridge')
print(f'  MARP_rep  : {dict(zip(asset_labels, strategies["MARP_rep"].round(4)))}')

# 5. HRP (if available)
try:
    from src.optimizer import allocate_hrp
    strategies['HRP_5'] = allocate_hrp(ret_is, vol_target=0.05)
    print(f'  HRP_5    : {dict(zip(asset_labels, strategies["HRP_5"].round(4)))}')
except ImportError:
    print('  HRP_5    : [module not yet available]')

In [ ]:
# Weight comparison bar chart
weight_df = pd.DataFrame(strategies).T
weight_df.columns = asset_labels

fig, ax = plt.subplots(figsize=(12, 5))
x = np.arange(len(weight_df))
width = 0.15
colors = ['#2196F3', '#4CAF50', '#FF9800', '#F44336', '#9C27B0']

for i, (asset, color) in enumerate(zip(asset_labels, colors)):
    ax.bar(x + i * width, weight_df[asset], width, label=asset, color=color, alpha=0.85)

ax.set_xticks(x + width * 2)
ax.set_xticklabels(weight_df.index, fontsize=10)
ax.set_ylabel('Weight')
ax.set_title('Strategy Weight Comparison (In-Sample 2017–2021)')
ax.legend(fontsize=9, ncol=5, loc='upper center', bbox_to_anchor=(0.5, -0.12))
ax.yaxis.set_major_formatter(mticker.PercentFormatter(xmax=1.0))
fig.tight_layout()
plt.show()

In [ ]:
# Risk contribution analysis for ERC portfolio
from src.optimizer import _risk_contributions

erc_w = strategies['ERC_5'].values
rc = _risk_contributions(ret_is, erc_w)

fig, axes = plt.subplots(1, 3, figsize=(14, 4.5))

# ERC risk contributions (should be equalised)
axes[0].barh(asset_labels, rc / rc.sum() * 100, color=colors, alpha=0.8)
axes[0].axvline(x=100/5, color='black', linestyle='--', linewidth=1.0, label='Equal (20%)')
axes[0].set_title('ERC: Risk Contribution per Asset')
axes[0].set_xlabel('% of Total Portfolio Risk')
axes[0].legend(fontsize=8)

# Inverse vol weights vs risk contribution
rp_w = strategies['RP_5'].values
rp_rc = _risk_contributions(ret_is, rp_w)
axes[1].barh(asset_labels, rp_rc / rp_rc.sum() * 100, color=colors, alpha=0.8)
axes[1].set_title('RP_5: Risk Contribution per Asset')
axes[1].set_xlabel('% of Total Portfolio Risk')

# Covariance structure
cov_ann = ret_is.cov() * 252 * 10000  # annualised, scaled to bps² for readability
cov_ann.index = asset_labels
cov_ann.columns = asset_labels
sns.heatmap(cov_ann, annot=True, fmt='.0f', cmap='YlOrRd', ax=axes[2],
            cbar_kws={'label': 'bps²', 'shrink': 0.7})
axes[2].set_title('Annualised Covariance (bps²)')

fig.tight_layout()
plt.show()

In [ ]:
# Rolling ERC weights over time (in-sample)
window = 504  # 2 years
step = 126    # re-estimate every 6 months

rolling_weights = []
rolling_dates = []

for end in range(window, len(ret_is), step):
    window_ret = ret_is.iloc[end - window:end]
    w = allocate_erc(window_ret, vol_target=0.05)
    rolling_weights.append(w)
    rolling_dates.append(ret_is.index[end])

rw_df = pd.DataFrame(rolling_weights, index=rolling_dates)
rw_df.columns = asset_labels

fig, ax = plt.subplots(figsize=(14, 5))
for i, (asset, color) in enumerate(zip(asset_labels, colors)):
    ax.fill_between(rw_df.index, 0, rw_df[asset], label=asset, color=color, alpha=0.7, linewidth=0.5)
ax.set_title('Rolling ERC Weights (2yr window, re-estimated semi-annually)')
ax.set_ylabel('Weight')
ax.yaxis.set_major_formatter(mticker.PercentFormatter(xmax=1.0))
ax.legend(fontsize=9, ncol=5, loc='upper center', bbox_to_anchor=(0.5, -0.12))
fig.tight_layout()
plt.show()

In [ ]:
# MARP replication: tracking error decomposition
marp_w = strategies['MARP_rep']
replicated_ret = (ret_is[marp_w.index] * marp_w.values).sum(axis=1, skipna=True)
replicated_ret = replicated_ret.dropna()

aligned_marp = marp_is.reindex(replicated_ret.index).dropna()
aligned_rep = replicated_ret.reindex(aligned_marp.index)

tracking_diff = aligned_rep - aligned_marp

fig, axes = plt.subplots(2, 1, figsize=(14, 7))

rep_cum = (1 + aligned_rep).cumprod()
marp_cum = (1 + aligned_marp).cumprod()
axes[0].plot(rep_cum.index, rep_cum, color='#2196F3', linewidth=1.2, label='MARP Replication')
axes[0].plot(marp_cum.index, marp_cum, color='black', linewidth=1.2, linestyle='--', label='CSI MARP 930929')
axes[0].set_title('MARP Replication — Cumulative Returns (In-Sample)')
axes[0].legend(fontsize=9)
axes[0].set_ylabel('Cumulative Return')

axes[1].fill_between(tracking_diff.index, 0, tracking_diff * 100, alpha=0.3, color='#F44336')
axes[1].plot(tracking_diff.index, tracking_diff * 100, color='#F44336', linewidth=0.5)
axes[1].set_title('Daily Tracking Error (%)')
axes[1].set_ylabel('Tracking Error (%)')

te_annual = tracking_diff.std() * np.sqrt(252) * 100
print(f'Annualised Tracking Error: {te_annual:.2f}%')
print(f'Correlation: {aligned_rep.corr(aligned_marp):.4f}')
print(f'R²: {aligned_rep.corr(aligned_marp)**2:.4f}')

fig.tight_layout()
plt.show()

In [ ]:
# Vol-target calibration check
for name in ['ERC_5', 'RP_5']:
    if name in strategies:
        w = strategies[name].values
        port_ret = (ret_is.iloc[:, w > 0] * w[w > 0]).sum(axis=1)
        actual_vol = port_ret.std() * np.sqrt(252)
        print(f'{name}: target_vol=5.0%, actual_vol={actual_vol*100:.1f}%')

# Equal weight for reference
if 'Equal' in strategies:
    w = strategies['Equal'].values
    port_ret = (ret_is * w).sum(axis=1)
    actual_vol = port_ret.std() * np.sqrt(252)
    print(f'Equal: actual_vol={actual_vol*100:.1f}%')

## Key Observations

- ERC assigns higher weights to low-vol assets (bonds, gold) while maintaining risk balance
- MARP replication via ridge regression captures the index's factor exposures
- Vol-target scaling ensures consistent risk budget across strategies
- HRP should theoretically improve robustness to estimation errors in covariance